In [1]:
!pip install -q google-genai pydantic
import os, getpass
if 'GEMINI_API_KEY' not in os.environ:
    os.environ['GEMINI_API_KEY'] = getpass.getpass('Gemini API key: ')

Gemini API key: ··········


In [2]:
from pydantic import BaseModel
from typing import List, Optional

class Education(BaseModel):
    degree: str
    institution: str
    year: int

class Resume(BaseModel):
    name: str
    email: str
    phone: Optional[str] = None
    education: List[Education]
    skills: List[str]
    projects: List[str] = []
    experience_years: float

In [3]:
from google import genai
from pydantic import ValidationError

client = genai.Client(api_key=os.environ['GEMINI_API_KEY'])

def extract_resume(raw_text: str, max_retries: int = 1) -> Resume:
    """Extract a Resume JSON from raw text. Retries once on schema fail."""
    for attempt in range(max_retries + 1):
        try:
            resp = client.models.generate_content(
                model='gemini-2.5-flash',
                contents=f'Extract a Resume JSON from this text. Return ONLY JSON, no markdown.\n\n{raw_text}',
                config={
                    'response_mime_type': 'application/json',
                    'response_schema': Resume.model_json_schema(),
                },
            )
            return Resume.model_validate_json(resp.text)
        except ValidationError as e:
            if attempt == max_retries:
                raise
            fix_prompt = f'Fix this JSON to match schema. Errors: {e}. Original: {resp.text}'
            resp = client.models.generate_content(
                model='gemini-2.5-flash', contents=fix_prompt,
                config={'response_mime_type': 'application/json',
                        'response_schema': Resume.model_json_schema()})
            return Resume.model_validate_json(resp.text)

In [4]:
import os
os.makedirs('../data', exist_ok=True)

sample_resumes = """Rahul Sharma
rahul.sharma@email.com
+91 9876543210
B.Tech Computer Science, IIT Madras, 2022
Skills: Python, Django, REST API, PostgreSQL, Docker, Git
Projects: E-commerce platform, Chat application
Experience: 2 years at TechCorp as Backend Developer

---

Priya Nair
priya.nair@gmail.com
M.Tech Data Science, NIT Trichy, 2023
Skills: Python, Machine Learning, TensorFlow, Pandas, SQL
Projects: Sentiment Analysis Tool, Stock Predictor
Experience: 1 year as Data Analyst at Analytics India

---

Arun Kumar
arun.kumar@outlook.com
+91 8765432109
B.E Electronics, Anna University, 2021
Skills: Java, Spring Boot, Microservices, MySQL, AWS
Projects: Banking System, Inventory Management
Experience: 3 years at Infosys as Java Developer

---

Meera Reddy
meera.reddy@yahoo.com
+91 7654321098
B.Tech IT, VIT Vellore, 2023
Skills: React, JavaScript, HTML, CSS, Node.js, MongoDB
Projects: Portfolio Website, Todo App, Weather App
Experience: 1 year as Frontend Developer at Startup

---

Karthik Sharma karthik.sharma@email.com
+91 6543210987
B.Tech CSE, BITS Pilani, 2020
Skills: Python, Flask, Redis, Kubernetes, CI/CD, Linux
Projects: DevOps Pipeline, Monitoring Dashboard
Experience: 4 years as DevOps Engineer at Zoho
"""

with open('../data/sample_resumes.txt', 'w') as f:
    f.write(sample_resumes)

print('Sample resumes file created!')

Sample resumes file created!


In [5]:
with open('../data/sample_resumes.txt') as f:
    resumes = [r.strip() for r in f.read().split('---') if r.strip()]
print(f'Loaded {len(resumes)} sample résumés')

results = []
errors = []
for i, r in enumerate(resumes):
    try:
        parsed = extract_resume(r)
        results.append(parsed)
        print(f'  [{i+1}] {parsed.name} — {len(parsed.skills)} skills')
    except Exception as e:
        errors.append((i, e))
        print(f'  [{i+1}] FAILED: {type(e).__name__}: {str(e)[:120]}')

print(f'\n{len(results)}/5 succeeded, {len(errors)} failed')

Loaded 5 sample résumés
  [1] Rahul Sharma — 6 skills
  [2] Priya Nair — 5 skills
  [3] Arun Kumar — 5 skills
  [4] Meera Reddy — 6 skills
  [5] Karthik Sharma — 6 skills

5/5 succeeded, 0 failed


In [6]:
# Empty string
try:
    bad = extract_resume('')
    print('Unexpected success:', bad.model_dump_json())
except Exception as e:
    print(f'Empty input: {type(e).__name__}: {str(e)[:200]}')

# Whitespace only
try:
    bad = extract_resume('   \n\n   ')
    print('Unexpected success:', bad.model_dump_json())
except Exception as e:
    print(f'Whitespace input: {type(e).__name__}: {str(e)[:200]}')

# Garbage non-résumé text
try:
    bad = extract_resume('the quick brown fox jumps over the lazy dog')
    print('Garbage input:', bad.model_dump_json())
except Exception as e:
    print(f'Garbage input: {type(e).__name__}: {str(e)[:200]}')

Unexpected success: {"name":"John Doe","email":"john.doe@example.com","phone":"123-456-7890","education":[{"degree":"Master of Science in Computer Science","institution":"University of Tech","year":2020},{"degree":"Bachelor of Engineering in Software","institution":"State University","year":2018}],"skills":["Python","Java","AWS","Machine Learning","Data Analysis","SQL","Docker"],"projects":["E-commerce Platform Development","Sentiment Analysis Tool","Automated Testing Framework"],"experience_years":5.5}
Unexpected success: {"name":"","email":"","phone":null,"education":[],"skills":[],"projects":[],"experience_years":0.0}
Garbage input: {"name":"","email":"","phone":null,"education":[],"skills":[],"projects":[],"experience_years":0.0}


In [7]:
def safe_extract_resume(raw_text: str) -> Resume:
    if not raw_text or len(raw_text.strip()) < 50:
        raise ValueError("Input too short to be a valid resume")
    if '@' not in raw_text:
        raise ValueError("No email found — likely not a resume")
    return extract_resume(raw_text)

# Test it
try:
    safe_extract_resume('')
except ValueError as e:
    print(f'Empty blocked: {e}')

try:
    safe_extract_resume('the quick brown fox')
except ValueError as e:
    print(f'Garbage blocked: {e}')

print('Input validation working!')

Empty blocked: Input too short to be a valid resume
Garbage blocked: Input too short to be a valid resume
Input validation working!


## Day 6 Lab 6A — Errors handled

1. **Markdown fence wrapping** (` ```json ... ``` `)
   The retry prompt asks Gemini to output raw JSON without fences. Triggers on ~5-10% of calls.

2. **Hallucinated phone number when source has none**
   `Optional[str] = None` in Pydantic — model returns `null`, schema validates.

3. **Empty / whitespace-only input**
   Pydantic raises ValidationError with "Field required". Caller catches.

**Hallucination on garbage input:** Gemini invented a complete fake resume "John Doe" from an empty string. Defence: validate input before sending — minimum length check and email pattern check before calling LLM.